In [1]:
import requests

import time
import pandas as pd
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service


In [2]:
from selenium.common.exceptions import NoSuchElementException

In [3]:
#FILE_PATH_FOLDER = 'F:....Blogathon'
search_query = 'https://comprar.gob.ar/BuscarAvanzado.aspx'#'https://www.indeed.com/q-data-scientist-jobs.html'
driver = webdriver.Chrome(executable_path='C:/chromedriver/chromedriver.exe')

In [4]:
driver.get(search_query)

In [5]:
#grabar cada linea en un csv
def guarda_datos_items():
    #print(contratos_details)
    print('guarda datos items')
    datos=pd.DataFrame(items_details, columns =['num_expediente', 'num_proceso','objeto_del_gasto','codigo_del_item','descripcion', 'cantidad' ])
    #print(datos)
    datos.to_csv(path_or_buf='D:/comprar/unificados/diferencia/iprocesos_items.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)


In [11]:
lista_procesos = pd.read_csv("D:/comprar/unificados/difItem2022.csv", sep='|')
lista_procesos=lista_procesos[lista_procesos.columns[0]]
lista_procesos

0        334-0001-SPU21
1      37/13-0001-SPU22
2      37/28-0046-CDI22
3       38/2-0256-LPU22
4       38/2-0257-LPU22
             ...       
139    84/13-0102-SPU22
140    84/13-0103-SPU22
141    84/13-0104-SPU22
142    84/13-0112-SPU22
143       92-0024-CDI22
Name: num_proceso, Length: 144, dtype: object

In [21]:
#inicializa dataframes
#print('inicializa dataframes')

clases_details = []
clases_doc_info=[]

items_details = []
items_info=[]


In [7]:
def extrae_datos_items():
    num_exp = driver.find_element_by_id('ctl00_CPH1_UCVistaPreviaPliego_usrCabeceraPliego_lblNumExpediente')
    print(num_exp.text)
    num_proceso = driver.find_element_by_id('ctl00_CPH1_UCVistaPreviaPliego_usrCabeceraPliego_lblNumPliego')
    print(num_proceso.text)
    try:
        tabla_items = driver.find_element_by_id('ctl00_CPH1_UCVistaPreviaPliego_UC_DetalleProductosOCA_gvItemsOCA')
        id = 'ctl00_CPH1_UCVistaPreviaPliego_UC_DetalleProductosOCA_gvItemsOCA'         
    except Exception:
        tabla_items=''
        print('no está la tabla items ¿1? - pruebo otro id')
        try:
            tabla_items = driver.find_element_by_id('ctl00_CPH1_UCVistaPreviaPliego_UC_DetalleProductosCM_gvItemsCM')
            id = 'ctl00_CPH1_UCVistaPreviaPliego_UC_DetalleProductosCM_gvItemsCM'              
        except Exception:
            tabla_items=''
            print('no está la tabla items ¿2? - pruebo otro id')
            try:
                tabla_items = driver.find_element_by_id('ctl00_CPH1_UCVistaPreviaPliego_UC_DetalleProductos_gvLineaPliego')
                id = 'ctl00_CPH1_UCVistaPreviaPliego_UC_DetalleProductos_gvLineaPliego'
            except Exception:
                tabla_items=''
                print('no está la tabla items ¿3? - pruebo otro id')
                try:
                    tabla_items = driver.find_element_by_id('ctl00_CPH1_UCVistaPreviaPliego_UC_DetalleLotesSubastaPublica_gvLotes')
                    id = 'ctl00_CPH1_UCVistaPreviaPliego_UC_DetalleLotesSubastaPublica_gvLotes'
                except Exception:
                    tabla_items=''
                    print('no está la tabla items ¿fin? - pruebo otro id')
    print(tabla_items)                 
    if tabla_items!='': 
        rows = tabla_items.find_elements_by_xpath('//*[@id="'+id+'"]/tbody/tr')
        number_of_rows = len(rows)
        print('nro de rows a iterar dentro de la tabla : '+str(number_of_rows))
        for row in rows:
            # Get the columns(all the column 2)
            cols = row.find_elements_by_tag_name("td")
            number_of_cols = len(cols)
            #print(number_of_cols)
            #note: index start from 0, 1 is col 2
            if number_of_cols > 0:
                try:
                    objeto_del_gasto=cols[1].text 
                except Exception:
                    objeto_del_gasto=''
                    print('error nro objeto_del_gasto')
                try:
                    codigo_del_item = cols[2].text
                except Exception:
                    codigo_del_item=''
                    print('error codigo_del_item')
                #print(codigo_del_item)    
                try:
                    descripcion = cols[3].text.replace(';',' -').replace("\n", " ")
                except Exception:
                    descripcion=''
                    print('error descripcion')
                #print(descripcion)  
                try:
                    cantidad = cols[4].text
                except Exception:
                    cantidad=''
                    print('error cantidad')
                #print(cantidad)
                items_info = [num_exp.text,num_proceso.text,objeto_del_gasto,codigo_del_item,descripcion, cantidad]
                items_details.append(items_info)    
    print('sale items! al fin')

In [8]:
def obtiene_datos_pag(): #agregar un try catch por la cantidad de elementos menor a 10
    
        try:
            print('--------------items----------')
            extrae_datos_items()
       
            print('volver')
            link_volver = driver.find_element_by_id('ctl00_CPH1_lnkVolver')
            link_volver.click()   
            time.sleep(random.uniform(5.0,10.0))#20, 25
                          
            guarda_datos_items()
        except Exception:
                print('nO FUNCIONA LINK PARA ESTE ITEM')
                driver.execute_script("window.history.go(-1)")
                
       
        

In [54]:
#si el buscar devuelve una sola fila

clases_details = []
clases_doc_info=[]

items_details = []
items_info=[]

for num_proceso in lista_procesos: 
    print('-------------------------------------------------------------------------')
    print(num_proceso)
    
    #carga la página y le pone los parámetros
    driver.get(search_query)
    time.sleep(random.uniform(3.0,5.0))   
    driver.find_element_by_id('ctl00_CPH1_txtNumeroProceso').clear()
    input_nro_proceso = driver.find_element_by_id('ctl00_CPH1_txtNumeroProceso')
    #print(input_cuit_proveedor)
    input_nro_proceso.send_keys(num_proceso)
    btn_nro_proceso=  driver.find_element_by_id('ctl00_CPH1_btnListarPliegoNumero') 
    btn_nro_proceso.click()
    time.sleep(random.uniform(1.0,2.0)) 
         
    try:
        link_pagina = driver.find_element_by_id('ctl00_CPH1_GridListaPliegos_ctl02_lnkNumeroProceso')
        link_pagina.click() 
        time.sleep(random.uniform(1.0,2.0))           
        obtiene_datos_pag()              
        pausaNP = 1
        print('ok tiempo carga next page!')
    except Exception:
        time.sleep(random.uniform(1.0,2.0)) 
        print('MAS TIEMPO para carga next page!')

-------------------------------------------------------------------------
16-0001-CPU17
--------------items----------
EX-2017-14414899-   -APN-GAAYR#ARN
16-0001-CPU17
<selenium.webdriver.remote.webelement.WebElement (session="8ac41a70427d437915dcf3806e5447fa", element="8a98ec4a-4964-4c5a-a243-3114daf88994")>
nro de rows a iterar dentro de la tabla : 1
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
-------------------------------------------------------------------------
16-0060-CDI17
--------------items----------
EX-2017-08220990-   -APN-GAAYR#ARN
16-0060-CDI17
<selenium.webdriver.remote.webelement.WebElement (session="8ac41a70427d437915dcf3806e5447fa", element="a7a72a53-ea85-454d-a117-a8c175ff2fdc")>
nro de rows a iterar dentro de la tabla : 1
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
-------------------------------------------------------------------------
16-0080-CDI17
--------------items----------
EX-2017-11083840-   -APN-GAAYR

KeyboardInterrupt: 

In [16]:
#si el buscar devuelve mas de una fila
clases_details = []
clases_doc_info=[]

items_details = []
items_info=[]

for num_proceso in lista_procesos: 
    print('-------------------------------------------------------------------------')
    print(num_proceso)
    #num_proceso = '334-0001-SPU18'
    nro_casos = 0
    #carga la página y le pone los parámetros
    driver.get(search_query)
    time.sleep(random.uniform(3.0,5.0))   
    driver.find_element_by_id('ctl00_CPH1_txtNumeroProceso').clear()
    input_nro_proceso = driver.find_element_by_id('ctl00_CPH1_txtNumeroProceso')
    #print(input_cuit_proveedor)
    input_nro_proceso.send_keys(num_proceso)
    btn_nro_proceso=  driver.find_element_by_id('ctl00_CPH1_btnListarPliegoNumero') 
    btn_nro_proceso.click()
    time.sleep(random.uniform(1.0,2.0)) 
    
    
    #obtiene texto para ver el nro de casos
    texto_nro_casos = driver.find_element_by_id('ctl00_CPH1_lblCantidadListaPliegos')
    print(texto_nro_casos.text)
    indice1 = texto_nro_casos.text.find("(")
    print(indice1)
    indice2=texto_nro_casos.text.find(")")
    print(indice2)
    #como cada página tiene 10 lineas se obtiene el nro de páginas a iterar
    nro_casos=texto_nro_casos.text[indice1+1:indice2]
    print('nro_casos '+str(nro_casos))
    nro_casos = int(nro_casos)
    i=1 #hojas totales
    indice=2 # por tabla max 11
    
        #for i in range (2, nro_casos):
    while i <= nro_casos:
        print('entra while')
        i += 1  
        try:
            link_pagina = driver.find_element_by_id('ctl00_CPH1_GridListaPliegos_ctl0'+str(indice)+'_lnkNumeroProceso')                                                    
            print(link_pagina)                                                   
            link_pagina.click() 
            time.sleep(random.uniform(1.0,2.0))           
            obtiene_datos_pag()              
            print('ok tiempo carga next page!')                                  
        except Exception:
            print('Error en la página: '+str(i)+driver.find_element_by_css_selector('#ctl00_CPH1_GridListaPliegos > tbody > tr.pagination-gv > td > table > tbody > tr > td:nth-child('+str(indice)+') > a'))
   
        indice = indice +1
        print('indice: '+str(indice))
        print('i: '+str(i))
        print('------------------------------------------------------------------------------')

         
    

-------------------------------------------------------------------------
334-0001-SPU21
Se han encontrado (1) resultados para su búsqueda
18
20
nro_casos 1
entra while
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="8378f3f9-c432-4b5e-ba54-cb3258904a32")>
--------------items----------
EX-2021-117105334-   -APN-DCOYCO#MAGYP
334-0001-SPU21
no está la tabla items ¿1? - pruebo otro id
no está la tabla items ¿2? - pruebo otro id
no está la tabla items ¿3? - pruebo otro id
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="21df1816-6f11-4019-b3af-953dfc0b21d3")>
nro de rows a iterar dentro de la tabla : 1
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
------------------------------------------------------------------------------
-------------------------------------------------------------------------
37/13-0001-SPU22
Se han encontrado (1) result

--------------items----------
EX-2022-18701784-   -APN-DCCYS#AABE
392-0005-SPU22
no está la tabla items ¿1? - pruebo otro id
no está la tabla items ¿2? - pruebo otro id
no está la tabla items ¿3? - pruebo otro id
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="643cc99b-a16b-44be-ba20-b4ae9db25c6d")>
nro de rows a iterar dentro de la tabla : 1
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
------------------------------------------------------------------------------
-------------------------------------------------------------------------
392-0009-SPU22
Se han encontrado (1) resultados para su búsqueda
18
20
nro_casos 1
entra while
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="de6c3054-83c7-4cff-9b41-babf814b748e")>
--------------items----------
EX-2022-18732378-   -APN-DCCYS#AABE
392-0009-SPU22
no está la tabla items ¿1? - pruebo otro 

Se han encontrado (1) resultados para su búsqueda
18
20
nro_casos 1
entra while
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="1dfe689f-3e98-4e1b-9dc6-1da8b732287c")>
--------------items----------
EX-2022-106347845-   -APN-DCYC#MC
83-0110-CDI22
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="becdb1b7-e685-4182-881e-4425e8f65727")>
nro de rows a iterar dentro de la tabla : 1
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
------------------------------------------------------------------------------
-------------------------------------------------------------------------
84/11-0917-LPR22
Se han encontrado (1) resultados para su búsqueda
18
20
nro_casos 1
entra while
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="2c72235d-356a-4754-9e8e-5f4a3a1dcfd8")>
--------------items----------
EX

guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
------------------------------------------------------------------------------
-------------------------------------------------------------------------
84/13-0011-SPU22
Se han encontrado (1) resultados para su búsqueda
18
20
nro_casos 1
entra while
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="6b77f965-f94f-45a5-a410-9f5a394440db")>
--------------items----------
EX-2022-15089379-   -APN-DRV#EA
84/13-0011-SPU22
no está la tabla items ¿1? - pruebo otro id
no está la tabla items ¿2? - pruebo otro id
no está la tabla items ¿3? - pruebo otro id
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="2d61bf9a-8176-4cfe-9617-efe8c42ff25b")>
nro de rows a iterar dentro de la tabla : 3
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
-------------------------------------------------------------

--------------items----------
EX-2022-39152281-   -APN-DRV#EA
84/13-0025-SPU22
no está la tabla items ¿1? - pruebo otro id
no está la tabla items ¿2? - pruebo otro id
no está la tabla items ¿3? - pruebo otro id
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="ad48e263-867c-4cf7-b998-f45b2b469e22")>
nro de rows a iterar dentro de la tabla : 1
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
------------------------------------------------------------------------------
-------------------------------------------------------------------------
84/13-0026-SPU22
Se han encontrado (1) resultados para su búsqueda
18
20
nro_casos 1
entra while
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="30efff76-b4bf-41b0-bde6-3b12e200c401")>
--------------items----------
EX-2022-40429230-   -APN-DRV#EA
84/13-0026-SPU22
no está la tabla items ¿1? - pruebo otro id

...

--------------items----------
EX-2022-126669288-   -APN-DRV#EA
84/13-0111-SPU22
no está la tabla items ¿1? - pruebo otro id
no está la tabla items ¿2? - pruebo otro id
no está la tabla items ¿3? - pruebo otro id
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="9654368c-a7f0-4611-8996-dded7de9405d")>
nro de rows a iterar dentro de la tabla : 2
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
------------------------------------------------------------------------------
-------------------------------------------------------------------------
84/136-1941-LPR21
Se han encontrado (1) resultados para su búsqueda
18
20
nro_casos 1
entra while
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="985e1eeb-2be0-4eb8-944b-e2d5c7f46840")>
--------------items----------
EX-2021-118283665-   -APN-CGMBA#EA
84/136-1941-LPR21
<selenium.webdriver.remote.webelement

--------------items----------
EX-2021-121672299-   -APN-DEOP#EA
84/88-2031-LPR21
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="bebdb6ea-df44-4840-8cea-f3e098ff51bf")>
nro de rows a iterar dentro de la tabla : 2
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
------------------------------------------------------------------------------
-------------------------------------------------------------------------
84/98-0592-LPR22
Se han encontrado (1) resultados para su búsqueda
18
20
nro_casos 1
entra while
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="1ae8d58c-d197-48e6-94f6-66161b23abf3")>
--------------items----------
EX-2022-35185436-   -APN-RIUP#EA
84/98-0592-LPR22
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="571b75cd-c585-4755-9787-877c4827ea79")>
nro de rows a iterar dentro d

--------------items----------
EX-2022-72790800-   -APN-DCON#FAA
40/47-0608-CDI22
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="f3b08b1b-7c06-44da-9adc-9fd0ba799b8b")>
nro de rows a iterar dentro de la tabla : 2
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
------------------------------------------------------------------------------
-------------------------------------------------------------------------
72-0011-LPU22
Se han encontrado (1) resultados para su búsqueda
18
20
nro_casos 1
entra while
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="14c73f89-c21c-4692-8b25-21887cd1a2c0")>
--------------items----------
EX-2022-120395933-   -APN-DAF#EMCO
72-0011-LPU22
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="29834156-334f-4a88-922c-b35cfd8818f0")>
nro de rows a iterar dentro de la

nro de rows a iterar dentro de la tabla : 1
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
------------------------------------------------------------------------------
-------------------------------------------------------------------------
84/20-0830-LPR22
Se han encontrado (1) resultados para su búsqueda
18
20
nro_casos 1
entra while
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="f99cff2d-bba4-4695-b63b-47e5b75e0605")>
--------------items----------
EX-2022-48521819-   -APN-CBMIX#EA
84/20-0830-LPR22
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="9dead7bd-0367-4d37-bd8c-f8a1a3ae824b")>
nro de rows a iterar dentro de la tabla : 2
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
------------------------------------------------------------------------------
------------------------------------------

sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
------------------------------------------------------------------------------
-------------------------------------------------------------------------
84/13-0102-SPU22
Se han encontrado (1) resultados para su búsqueda
18
20
nro_casos 1
entra while
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="a94d7bee-56ff-498b-9ce7-78a26d225ff0")>
--------------items----------
EX-2022-117459807-   -APN-DRV#EA
84/13-0102-SPU22
no está la tabla items ¿1? - pruebo otro id
no está la tabla items ¿2? - pruebo otro id
no está la tabla items ¿3? - pruebo otro id
<selenium.webdriver.remote.webelement.WebElement (session="cd030ce170f0902ee0f92f4ad41ce125", element="049761bb-ed6a-4276-b57a-9bbb5b8e5c2e")>
nro de rows a iterar dentro de la tabla : 9
sale items! al fin
volver
guarda datos items
ok tiempo carga next page!
indice: 3
i: 2
----------------------------------

In [10]:
#Clases inscriptas - Obtiene los rubros
def obtiene_clases():
    print('entra obtiene datos clases')
    CUIT=driver.find_element_by_id("ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNumeroCUIT").text.replace('-', '')
    
    tabla_rubros = driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvClasesInscriptas')
    rows = tabla_rubros.find_elements_by_xpath('//*[@id="ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvClasesInscriptas"]/tbody/tr')
    number_of_rows = len(rows)
    print('nro de rows a iterar dentro de la tabla Rubros: '+str(number_of_rows))
    for row in rows:
        # Get the columns(all the column 2)
        cols = row.find_elements_by_tag_name("td")
        number_of_cols = len(cols)
        #print(number_of_cols)
        if number_of_cols > 0:
            try:
                codigo_clase=cols[0].text 
            except Exception:
                codigo_clase=''
                print('error codigo clase')
            #print(codigo_clase)
            try:
                descripcion_clase= cols[1].text.replace(';',' -').replace("\n", " ")
            except Exception:
                descripcion_clase=''
                print('error descripcion_clase')
            #print(descripcion_clase)    
            try:
                codigo_rubro = cols[2].text
            except Exception:
                codigo_rubro=''
                print('error codigo_rubro')
            #print(codigo_rubro)  
            try:
                descripcion_rubro = cols[3].text.replace(';',' -').replace("\n", " ")
            except Exception:
                descripcion_rubro=''
                print('error descripcion_rubro')
            #print(descripcion_rubro)
            clases_info = [CUIT, number_of_rows -1, codigo_clase,descripcion_clase,codigo_rubro,descripcion_rubro]
            clases_details.append(clases_info) 
            if(number_of_rows > 50):
                time.sleep(random.uniform(0.01,0.02)) 
            
    print('sale obtiene datos clases')
